# Final Goldset retrieval evaluation

This notebook runs the frozen 30-record comparison from `research/protocol/experiment-protocol.md`. It is separate from the development pilot. Do not change the task list, model, prompt, or retrieval settings after starting.

Run the code cells in order. Stop and share the output if a cell reports an error. The final run uses Qwen2.5-Coder-7B-Instruct snapshot `c03e6d358207e414f1eca0bb1891e29f1db0e242`; a smaller model cannot be substituted.

1. Check GPU and disk resources.
2. Download the project, pinned Goldset data, and the fixed task list.
3. Verify the 30 task tests before using the model.
4. Install model libraries and load the fixed model.
5. Run each valid task with retrieval off and on.
6. Review the paired summary and download the run-record archive.

The run records contain generated patches and test evidence. Keep the downloaded archive for the project report.


## 1. Check GPU and disk resources

In [ ]:
import json
import platform
import shutil
import subprocess
import sys
import torch

MODEL_ID = 'Qwen/Qwen2.5-Coder-7B-Instruct'
MODEL_REVISION = 'c03e6d358207e414f1eca0bb1891e29f1db0e242'
MAX_INPUT_TOKENS = 3072
MAX_NEW_TOKENS = 768
REQUIRED_GPU_FREE_GIB = 11.0
REQUIRED_DISK_FREE_GIB = 20.0

def gib(value):
    return round(value / (1024 ** 3), 2)

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('PyTorch CUDA build:', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU is required for this fixed-model evaluation. In Colab choose Runtime > Change runtime type > GPU, reconnect, and rerun this cell.')
props = torch.cuda.get_device_properties(0)
gpu_free_gib = gib(torch.cuda.mem_get_info(0)[0])
gpu_total_gib = gib(props.total_memory)
disk_free_gib = gib(shutil.disk_usage('/content').free)
print('GPU:', props.name)
print('GPU memory total / free (GiB):', gpu_total_gib, '/', gpu_free_gib)
print('Free disk at /content (GiB):', disk_free_gib)
try:
    import psutil
    memory = psutil.virtual_memory()
    ram_total_gib, ram_available_gib = gib(memory.total), gib(memory.available)
except ImportError:
    with open('/proc/meminfo', encoding='utf-8') as handle:
        memory_info = {line.split(':', 1)[0]: int(line.split()[1]) * 1024 for line in handle if ':' in line}
    ram_total_gib = gib(memory_info['MemTotal'])
    ram_available_gib = gib(memory_info.get('MemAvailable', memory_info['MemFree']))
print('System RAM total / available (GiB):', ram_total_gib, '/', ram_available_gib)
if gpu_free_gib < REQUIRED_GPU_FREE_GIB or disk_free_gib < REQUIRED_DISK_FREE_GIB:
    raise RuntimeError('Resources are below the frozen evaluation minimum. Stop here and share the complete output; do not load a different model.')
RESOURCE_REPORT = {
    'python': platform.python_version(), 'torch': torch.__version__,
    'cuda_build': torch.version.cuda, 'gpu': props.name,
    'gpu_total_gib': gpu_total_gib, 'gpu_free_gib': gpu_free_gib,
    'disk_free_gib': disk_free_gib, 'ram_total_gib': ram_total_gib,
    'ram_available_gib': ram_available_gib, 'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION, 'max_input_tokens': MAX_INPUT_TOKENS,
    'max_new_tokens': MAX_NEW_TOKENS,
}
print('Fixed evaluation model:', MODEL_ID)
print('Resource report JSON:', json.dumps(RESOURCE_REPORT, indent=2))


## 2. Download the project, pinned Goldset data, and task list

In [ ]:
import csv
import hashlib
import json
import subprocess
import sys
from pathlib import Path

PROJECT_URL = 'https://github.com/pisalaarti05/mtech-bug-fix-agent.git'
PROJECT_DIR = Path('/content/mtech-bug-fix-agent')
GOLDSET_DIR = Path('/content/goldset')
GOLDSET_REVISION = 'fe4f9bfbcb46637481242af2315c6b59951710b4'
if PROJECT_DIR.exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--quiet', PROJECT_URL, str(PROJECT_DIR)], check=True)
PROJECT_REVISION = subprocess.run(
    ['git', '-C', str(PROJECT_DIR), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True
).stdout.strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--editable', str(PROJECT_DIR)], check=True)

if not GOLDSET_DIR.exists():
    subprocess.run(['git', 'clone', '--quiet', '--no-checkout', 'https://github.com/andysalvo/goldset.git', str(GOLDSET_DIR)], check=True)
subprocess.run(['git', 'fetch', '--quiet', '--depth', '1', 'origin', GOLDSET_REVISION], cwd=GOLDSET_DIR, check=True)
subprocess.run(['git', 'checkout', '--quiet', '--detach', 'FETCH_HEAD'], cwd=GOLDSET_DIR, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'goldset'], capture_output=True, text=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--editable', str(GOLDSET_DIR), 'uv'], check=True)

manifest_path = PROJECT_DIR / 'research' / 'protocol' / 'goldset-evaluation-sample.csv'
manifest_bytes = manifest_path.read_bytes()
SAMPLE_SHA256 = hashlib.sha256(manifest_bytes).hexdigest()
manifest_rows = list(csv.DictReader(manifest_bytes.decode('utf-8').splitlines()))
if len(manifest_rows) != 30:
    raise RuntimeError(f'Expected exactly 30 frozen records, found {len(manifest_rows)}.')
if len({row['fingerprint'] for row in manifest_rows}) != 30 or len({row['repo'] for row in manifest_rows}) != 30:
    raise RuntimeError('The frozen sample must contain 30 unique records from 30 repositories.')
if any(row['goldset_revision'] != GOLDSET_REVISION for row in manifest_rows):
    raise RuntimeError('The task manifest and pinned Goldset revision do not match.')
records_by_fingerprint = {
    item['fingerprint']: item
    for item in (json.loads(line) for line in (GOLDSET_DIR / 'oracles.jsonl').read_text(encoding='utf-8').splitlines() if line.strip())
}
FINAL_RECORDS = []
for row in manifest_rows:
    item = records_by_fingerprint.get(row['fingerprint'])
    if item is None or item['repo'] != row['repo'] or item['commit'] != row['commit'] or item['parent'] != row['parent']:
        raise RuntimeError(f"Manifest record does not match the pinned Goldset corpus: {row['fingerprint']}")
    FINAL_RECORDS.append(item)
GOLDSET_SHA = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=GOLDSET_DIR, check=True, capture_output=True, text=True).stdout.strip()
RUN_DIR = PROJECT_DIR / 'artifacts' / 'runs' / 'goldset_final'
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'resource_report.json').write_text(json.dumps(RESOURCE_REPORT, indent=2) + '\n', encoding='utf-8')
print('Project revision:', PROJECT_REVISION)
print('Goldset revision:', GOLDSET_SHA)
print('Frozen records:', len(FINAL_RECORDS), 'from', len({item['repo'] for item in FINAL_RECORDS}), 'repositories')
print('Sample manifest SHA-256:', SAMPLE_SHA256)


## 3. Verify the 30 task tests before using the model

In [ ]:
import re
import subprocess
import sys

ORACLE_STATUS = {}
ORACLE_DETAILS = {}
for number, record in enumerate(FINAL_RECORDS, start=1):
    fingerprint = record['fingerprint']
    print(f"\n[{number}/30] {fingerprint} | {record['repo']} | {record['committed_at'][:10]}")
    checked = subprocess.run(
        [sys.executable, 'verify.py', '--corpus', 'oracles.jsonl', '--fingerprint', fingerprint],
        cwd=GOLDSET_DIR, capture_output=True, text=True,
    )
    if checked.stdout.strip():
        print(checked.stdout.strip())
    if checked.stderr.strip():
        print('Verifier message:', checked.stderr[-1000:])
    passed = checked.returncode == 0 and re.search(r'passed\s+1\s+failed\s+0\s+skipped\s+0', checked.stdout) is not None
    ORACLE_STATUS[fingerprint] = 'verified' if passed else 'not_verified'
    ORACLE_DETAILS[fingerprint] = {'status': ORACLE_STATUS[fingerprint], 'returncode': checked.returncode, 'stdout': checked.stdout, 'stderr': checked.stderr}
    print('Oracle status:', ORACLE_STATUS[fingerprint])
VERIFIED_RECORDS = [record for record in FINAL_RECORDS if ORACLE_STATUS[record['fingerprint']] == 'verified']
not_verified = [fingerprint for fingerprint, status in ORACLE_STATUS.items() if status != 'verified']
(RUN_DIR / 'oracle_verification.json').write_text(json.dumps(ORACLE_DETAILS, indent=2) + '\n', encoding='utf-8')
print(f"\nVerified: {len(VERIFIED_RECORDS)} / {len(FINAL_RECORDS)}")
if not_verified:
    print('These fixed sample records are preflight exclusions; no replacement tasks will be added:', not_verified)
if not VERIFIED_RECORDS:
    raise RuntimeError('No sample records passed Goldset verification. Stop and share this output before model evaluation.')


## 4. Install model libraries and load the fixed model

In [ ]:
import subprocess
import sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
    'transformers', 'accelerate', 'bitsandbytes'
], check=True)
from importlib.metadata import version
for package in ('transformers', 'accelerate', 'bitsandbytes'):
    print(f'{package}:', version(package))

from bugfix_agent.model import HuggingFaceModel
model = HuggingFaceModel(
    MODEL_ID,
    revision=MODEL_REVISION,
    max_input_tokens=MAX_INPUT_TOKENS,
    max_new_tokens=MAX_NEW_TOKENS,
)
if model.revision != MODEL_REVISION:
    raise RuntimeError('The loaded model revision does not match the frozen protocol.')
print('Model loaded:', model.model_id)
print('Immutable model snapshot:', model.revision)
print('Load time (seconds):', model.load_seconds)
print('GPU memory after load (GiB):', round(torch.cuda.memory_allocated() / (1024 ** 3), 2))


## 5. Run each task with retrieval off and on

In [ ]:
import json
import torch
from bugfix_agent.benchmark import run_goldset_record

if 'model' not in globals() or model.revision != MODEL_REVISION:
    raise RuntimeError('Run the fixed-model setup cell first; it must load the exact pinned 7B snapshot.')
if 'VERIFIED_RECORDS' not in globals():
    raise RuntimeError('Run the 30-record Goldset verification cell first.')

# Reuse completed records if a Colab runtime is interrupted and restarted.
completed = {}
for path in RUN_DIR.glob('*.json'):
    if path.name == 'resource_report.json' or path.name == 'summary.json':
        continue
    try:
        item = json.loads(path.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError):
        continue
    if (item.get('protocol_id') == 'goldset-public-30-v1'
        and item.get('model_revision') == MODEL_REVISION
        and item.get('benchmark_corpus_revision') == GOLDSET_SHA
        and item.get('sample_sha256') == SAMPLE_SHA256
        and item.get('protocol_code_revision') == 'd54f9fbeac811c62957ad3d36a64933e58caaf5d'):
        key = (item.get('benchmark_fingerprint'), item.get('condition'))
        if key not in completed or item.get('created_at_utc', '') > completed[key][1].get('created_at_utc', ''):
            completed[key] = (path, item)

for number, record in enumerate(VERIFIED_RECORDS, start=1):
    fingerprint = record['fingerprint']
    print(f"\nTASK {number}/{len(VERIFIED_RECORDS)}: {fingerprint} | {record['repo']}")
    for use_retrieval in (False, True):
        condition = 'retrieval' if use_retrieval else 'no_retrieval'
        key = (fingerprint, condition)
        if key in completed:
            print(condition, 'already recorded:', completed[key][1].get('test_status'))
            continue
        torch.cuda.empty_cache()
        result_path = run_goldset_record(
            record, model, RUN_DIR,
            use_retrieval=use_retrieval,
            top_k=3,
            corpus_revision=GOLDSET_SHA,
        )
        result = json.loads(result_path.read_text(encoding='utf-8'))
        result.update({
            'protocol_id': 'goldset-public-30-v1',
            'project_code_revision': PROJECT_REVISION,
            'sample_sha256': SAMPLE_SHA256,
            'protocol_code_revision': 'd54f9fbeac811c62957ad3d36a64933e58caaf5d',
        })
        result_path.write_text(json.dumps(result, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
        completed[key] = (result_path, result)
        print(condition, ':', result.get('test_status'))
        print('Buggy/reference checks:', result.get('buggy_revision_test_status'), '/', result.get('reference_fix_test_status'))
        if result.get('failing_tests'):
            print('Failing tests:', result['failing_tests'])
        if result.get('skip_reason'):
            print('Skip reason:', result['skip_reason'])
        print('Retrieved files:', [chunk.get('path') for chunk in result.get('retrieved_chunks', [])])
        print('Run record:', result_path)
print('All available task-condition runs are in:', RUN_DIR)


## 6. Review the paired results and download the run records

In [ ]:
import json
import math
import random
import statistics
import zipfile
from pathlib import Path

latest = {}
for path in RUN_DIR.glob('*.json'):
    if path.name in ('resource_report.json', 'summary.json'):
        continue
    try:
        item = json.loads(path.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError):
        continue
    if item.get('protocol_id') != 'goldset-public-30-v1' or item.get('sample_sha256') != SAMPLE_SHA256:
        continue
    key = (item.get('benchmark_fingerprint'), item.get('condition'))
    if key not in latest or item.get('created_at_utc', '') > latest[key][1].get('created_at_utc', ''):
        latest[key] = (path, item)

paired = []
for record in FINAL_RECORDS:
    fingerprint = record['fingerprint']
    no_item = latest.get((fingerprint, 'no_retrieval'), (None, {}))[1]
    yes_item = latest.get((fingerprint, 'retrieval'), (None, {}))[1]
    if no_item.get('test_status') in ('passed', 'failed') and yes_item.get('test_status') in ('passed', 'failed'):
        paired.append((fingerprint, int(no_item['test_status'] == 'passed'), int(yes_item['test_status'] == 'passed')))

n = len(paired)
no_pass = sum(row[1] for row in paired)
yes_pass = sum(row[2] for row in paired)
differences = [yes - no for _, no, yes in paired]
observed_difference = statistics.mean(differences) if differences else float('nan')
only_no = sum(no == 1 and yes == 0 for _, no, yes in paired)
only_yes = sum(no == 0 and yes == 1 for _, no, yes in paired)
discordant = only_no + only_yes
if discordant == 0:
    mcnemar_p = 1.0
else:
    tail = sum(math.comb(discordant, k) for k in range(min(only_no, only_yes) + 1)) / (2 ** discordant)
    mcnemar_p = min(1.0, 2 * tail)
if differences:
    rng = random.Random(20261124)
    boot = sorted(statistics.mean(rng.choices(differences, k=n)) for _ in range(10000))
    ci = [boot[int(0.025 * (len(boot) - 1))], boot[int(0.975 * (len(boot) - 1))]]
else:
    ci = [None, None]

scored_pairs = {fingerprint for fingerprint, _, _ in paired}
missing_pairs = len(FINAL_RECORDS) - len(scored_pairs)
summary = {
    'protocol_id': 'goldset-public-30-v1',
    'sample_sha256': SAMPLE_SHA256,
    'selected_records': len(FINAL_RECORDS),
    'paired_scorable_records': n,
    'unscored_or_incomplete_pairs': missing_pairs,
    'no_retrieval_passed': no_pass,
    'no_retrieval_rate': no_pass / n if n else None,
    'retrieval_passed': yes_pass,
    'retrieval_rate': yes_pass / n if n else None,
    'paired_rate_difference_retrieval_minus_no_retrieval': observed_difference if n else None,
    'paired_difference_bootstrap_95_percent_interval': ci,
    'no_retrieval_only_passes': only_no,
    'retrieval_only_passes': only_yes,
    'exact_mcnemar_two_sided_p': mcnemar_p,
}
summary['per_task'] = []
for record in FINAL_RECORDS:
    fingerprint = record['fingerprint']
    no_item = latest.get((fingerprint, 'no_retrieval'), (None, {}))[1]
    yes_item = latest.get((fingerprint, 'retrieval'), (None, {}))[1]
    summary['per_task'].append({
        'fingerprint': fingerprint, 'repository': record['repo'],
        'no_retrieval_status': no_item.get('test_status', 'missing'),
        'no_retrieval_failing_tests': no_item.get('failing_tests', []),
        'retrieval_status': yes_item.get('test_status', 'missing'),
        'retrieval_failing_tests': yes_item.get('failing_tests', []),
        'retrieved_files': sorted({chunk.get('path') for chunk in yes_item.get('retrieved_chunks', [])}),
    })
summary_path = RUN_DIR / 'summary.json'
summary_path.write_text(json.dumps(summary, indent=2) + '\n', encoding='utf-8')
print(json.dumps(summary, indent=2))
archive_path = Path('/content/goldset-final-run-records.zip')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIR.glob('*.json')):
        archive.write(path, arcname=path.name)
print('Run-record archive:', archive_path)
try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print('Download this archive from the Colab Files panel:', archive_path)
